# DeepCas13: exclude `N`, exact architecture, and 100 fixed-split trials

Run `DeepCas13_exclude_N_create_fixed_split.ipynb` first.

The splitting notebook removes every guide containing `N` before splitting.
This notebook verifies that the saved train, validation, and unseen subsets
contain only A/C/G/T sequences.

It retains:

- the authors' exact DeepCas13 architecture;
- exact sequence and ViennaRNA structure preprocessing;
- exact LFC-to-`y_value` transformation inherited from the split notebook;
- maximum 100 epochs;
- early stopping on validation loss with patience 10;
- 100 Optuna trials;
- validation MSE, unseen Pearson, and unseen Spearman.


In [ ]:
from pathlib import Path
import hashlib
import json
import random
import warnings

import numpy as np
import optuna
import pandas as pd
import RNA
import tensorflow as tf

from scipy.stats import pearsonr, spearmanr
from tensorflow.keras.layers import (
    BatchNormalization,
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    Input,
    LSTM,
    MaxPooling2D,
    TimeDistributed,
    concatenate,
)
from tensorflow.keras.models import Model

OUTPUT_DIR = Path(
    "results/deepcas13_exact_no_N_fixed_split"
)
SPLIT_DIR = OUTPUT_DIR / "saved_splits"
RESULTS_DIR = OUTPUT_DIR / "trial_results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = SPLIT_DIR / "train_split.csv"
VALIDATION_FILE = SPLIT_DIR / "validation_split.csv"
UNSEEN_FILE = SPLIT_DIR / "unseen_split.csv"
MANIFEST_FILE = OUTPUT_DIR / "split_manifest.json"

for path in [
    TRAIN_FILE,
    VALIDATION_FILE,
    UNSEEN_FILE,
    MANIFEST_FILE,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path.resolve()}. "
            "Run the split notebook first."
        )

MODEL_SEED = 42
OPTUNA_SEED = 42
N_TRIALS = 100

MAX_EPOCHS = 100
PATIENCE = 10

print("TensorFlow:", tf.__version__)


## Verify and load the saved split

In [ ]:
def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(MANIFEST_FILE, "r") as handle:
    manifest = json.load(handle)

assert (
    sha256(TRAIN_FILE)
    == manifest["train_sha256"]
)
assert (
    sha256(VALIDATION_FILE)
    == manifest["validation_sha256"]
)
assert (
    sha256(UNSEEN_FILE)
    == manifest["unseen_sha256"]
)

if not manifest.get(
    "N_filter_applied_before_split",
    False,
):
    raise ValueError(
        "The split manifest does not confirm that N-containing guides "
        "were excluded before splitting."
    )

train_data = pd.read_csv(
    TRAIN_FILE
)
validation_data = pd.read_csv(
    VALIDATION_FILE
)
unseen_data = pd.read_csv(
    UNSEEN_FILE
)

assert (
    len(train_data)
    == manifest["n_train"]
)
assert (
    len(validation_data)
    == manifest["n_validation"]
)
assert (
    len(unseen_data)
    == manifest["n_unseen"]
)

for subset_name, subset_frame in [
    ("train", train_data),
    ("validation", validation_data),
    ("unseen", unseen_data),
]:
    if subset_frame["seq"].astype(str).str.contains(
        "N",
        regex=False,
    ).any():
        raise ValueError(
            f"The saved {subset_name} subset contains a guide with N."
        )

print(
    "Original samples:",
    manifest["original_samples"],
)
print(
    "Excluded samples containing N:",
    manifest[
        "excluded_samples_containing_N"
    ],
)
print(
    "Subset sizes:",
    len(train_data),
    len(validation_data),
    len(unseen_data),
)


## Exact author preprocessing

In [ ]:
dct_ohc_seq = {
    "A": [1, 0, 0, 0],
    "C": [0, 1, 0, 0],
    "G": [0, 0, 1, 0],
    "T": [0, 0, 0, 1],
    "N": [0, 0, 0, 0],
}

dct_ohc_fold = {
    "(": [1, 0, 0],
    ")": [0, 1, 0],
    ".": [0, 0, 1],
    "N": [0, 0, 0],
}


def get_fold(seq):
    for _ in range(33 - len(seq)):
        seq = seq + "N"

    fold_compound = RNA.fold_compound(seq)
    mfe_structure, _ = fold_compound.mfe()

    return mfe_structure


def seq_one_hot_code(seq):
    seq = seq.upper()
    sequence_list = list(seq)

    sequence_list.extend(
        [
            "N"
            for _ in range(
                33 - len(seq)
            )
        ]
    )

    return [
        dct_ohc_seq[base]
        for base in sequence_list
    ]


def fold_one_hot_code(fold):
    fold_list = list(fold)

    fold_list.extend(
        [
            "N"
            for _ in range(
                33 - len(fold)
            )
        ]
    )

    return [
        dct_ohc_fold[symbol]
        for symbol in fold_list
    ]


def encode_frame(frame):
    sequences = [
        seq_one_hot_code(sequence)
        for sequence in frame["seq"].to_list()
    ]

    folds = [
        fold_one_hot_code(
            get_fold(sequence)
        )
        for sequence in frame["seq"].to_list()
    ]

    sequence_array = np.asarray(
        sequences,
        dtype=np.float32,
    )

    fold_array = np.asarray(
        folds,
        dtype=np.float32,
    )

    sequence_cnn = np.reshape(
        sequence_array,
        (
            len(sequence_array),
            1,
            33,
            4,
            1,
        ),
    )

    fold_cnn = np.reshape(
        fold_array,
        (
            len(fold_array),
            1,
            33,
            3,
            1,
        ),
    )

    labels = frame[
        "y_value"
    ].to_numpy(
        dtype=np.float32
    )

    return (
        sequence_cnn,
        fold_cnn,
        labels,
    )


X_train_seq, X_train_fold, y_train = encode_frame(
    train_data
)

(
    X_validation_seq,
    X_validation_fold,
    y_validation,
) = encode_frame(
    validation_data
)

X_unseen_seq, X_unseen_fold, y_unseen = encode_frame(
    unseen_data
)

print("Sequence input:", X_train_seq.shape)
print("Structure input:", X_train_fold.shape)
print("Label input:", y_train.shape)


## Exact author architecture

In [ ]:
def build_deepcas13_model(
    learning_rate=0.001,
):
    # Sequence branch
    seq_input = Input(
        shape=(1, 33, 4, 1)
    )

    seq_conv1 = TimeDistributed(
        Conv2D(
            8,
            (3, 3),
            padding="same",
            activation="relu",
        )
    )(seq_input)

    seq_norm1 = TimeDistributed(
        BatchNormalization()
    )(seq_conv1)

    seq_conv2 = TimeDistributed(
        Conv2D(
            16,
            (3, 3),
            padding="same",
            activation="relu",
        )
    )(seq_norm1)

    seq_norm2 = TimeDistributed(
        BatchNormalization()
    )(seq_conv2)

    seq_drop1 = TimeDistributed(
        Dropout(0.5)
    )(seq_norm2)

    seq_pool1 = TimeDistributed(
        MaxPooling2D((2, 2))
    )(seq_drop1)

    seq_flat1 = TimeDistributed(
        Flatten()
    )(seq_pool1)

    seq_lstm1 = LSTM(100)(
        seq_flat1
    )

    seq_drop2 = Dropout(0.3)(
        seq_lstm1
    )

    seq_output = Dense(
        64,
        activation="relu",
    )(seq_drop2)

    # Structure branch
    fold_input = Input(
        shape=(1, 33, 3, 1)
    )

    fold_conv1 = TimeDistributed(
        Conv2D(
            8,
            (3, 3),
            padding="same",
            activation="relu",
        )
    )(fold_input)

    fold_norm1 = TimeDistributed(
        BatchNormalization()
    )(fold_conv1)

    fold_conv2 = TimeDistributed(
        Conv2D(
            16,
            (3, 3),
            padding="same",
            activation="relu",
        )
    )(fold_norm1)

    fold_norm2 = TimeDistributed(
        BatchNormalization()
    )(fold_conv2)

    fold_drop1 = TimeDistributed(
        Dropout(0.5)
    )(fold_norm2)

    fold_pool1 = TimeDistributed(
        MaxPooling2D((2, 2))
    )(fold_drop1)

    fold_flat1 = TimeDistributed(
        Flatten()
    )(fold_pool1)

    fold_lstm1 = LSTM(100)(
        fold_flat1
    )

    fold_drop2 = Dropout(0.3)(
        fold_lstm1
    )

    fold_output = Dense(
        64,
        activation="relu",
    )(fold_drop2)

    # Fusion head
    merged = concatenate(
        [
            seq_output,
            fold_output,
        ],
        axis=1,
        name="merged",
    )

    dense_dropout = Dropout(0.3)(
        merged
    )

    dense_hidden = Dense(64)(
        dense_dropout
    )

    output = Dense(
        1,
        activation="sigmoid",
    )(dense_hidden)

    model = Model(
        [
            seq_input,
            fold_input,
        ],
        output,
    )

    if hasattr(tf.keras.optimizers, "legacy"):
        optimizer = tf.keras.optimizers.legacy.Adam(
            learning_rate=learning_rate
        )
    else:
        optimizer = tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        )

    model.compile(
        optimizer=optimizer,
        loss="mse",
    )

    return model


prototype_model = build_deepcas13_model()
prototype_model.summary()


## Metrics and reproducibility helpers

In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def safe_pearson(y_true, y_pred):
    if (
        len(y_true) < 2
        or np.std(y_true) == 0
        or np.std(y_pred) == 0
    ):
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(
            pearsonr(y_true, y_pred)[0]
        )


def safe_spearman(y_true, y_pred):
    if (
        len(y_true) < 2
        or np.std(y_true) == 0
        or np.std(y_pred) == 0
    ):
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(
            spearmanr(y_true, y_pred)[0]
        )


## Run 100 trials

Trial 0 uses the authors' exact defaults:

- learning rate: Adam default, 0.001
- batch size: 128

Other trials vary only learning rate and batch size. The architecture,
preprocessing, loss, epoch count, and shuffle behavior remain unchanged.


In [ ]:
trial_records = []

best_global_validation_mse = np.inf
best_global_trial = None

best_weights_file = (
    RESULTS_DIR
    / "best_DeepCas13.weights.h5"
)


def objective(trial):
    global best_global_validation_mse
    global best_global_trial

    set_all_seeds(MODEL_SEED)
    tf.keras.backend.clear_session()

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-5,
        3e-3,
        log=True,
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [32, 64, 128, 256],
    )

    model = build_deepcas13_model(
        learning_rate=learning_rate
    )

    history = model.fit(
        [
            X_train_seq,
            X_train_fold,
        ],
        y_train,
        validation_data=(
            [
                X_validation_seq,
                X_validation_fold,
            ],
            y_validation,
        ),
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        shuffle=True,
        verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)],
    )

    validation_losses = np.asarray(
        history.history["val_loss"],
        dtype=float,
    )

    best_epoch_index = int(
        np.argmin(validation_losses)
    )

    best_epoch = best_epoch_index + 1

    # EarlyStopping(restore_best_weights=True) has already restored
    # the weights from the epoch with the lowest validation loss.
    best_epoch_model = model

    validation_predictions = (
        best_epoch_model.predict(
            [
                X_validation_seq,
                X_validation_fold,
            ],
            verbose=0,
        )
        .reshape(-1)
    )

    validation_mse = float(
        np.mean(
            (
                y_validation
                - validation_predictions
            )
            ** 2
        )
    )

    unseen_predictions = (
        best_epoch_model.predict(
            [
                X_unseen_seq,
                X_unseen_fold,
            ],
            verbose=0,
        )
        .reshape(-1)
    )

    unseen_pearson = safe_pearson(
        y_unseen,
        unseen_predictions,
    )

    unseen_spearman = safe_spearman(
        y_unseen,
        unseen_predictions,
    )

    trial.set_user_attr(
        "best_epoch",
        int(best_epoch),
    )

    trial_records.append(
        {
            "trial": trial.number,
            "validation_mse": validation_mse,
            "unseen_pearson": unseen_pearson,
            "unseen_spearman": unseen_spearman,
        }
    )

    if validation_mse < best_global_validation_mse:
        best_global_validation_mse = validation_mse
        best_global_trial = trial.number

        best_epoch_model.save_weights(
            best_weights_file
        )

    print(
        f"Trial {trial.number:3d} | "
        f"lr={learning_rate:.3e} | "
        f"batch={batch_size:3d} | "
        f"Validation MSE: {validation_mse:.8f} | "
        f"Unseen Pearson: {unseen_pearson:.4f} | "
        f"Unseen Spearman: {unseen_spearman:.4f} | "
        f"Epoch: {best_epoch}"
    )

    return validation_mse


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(
        seed=OPTUNA_SEED
    ),
    study_name=(
        "DeepCas13_exact_architecture_"
        "100_trials"
    ),
)

study.enqueue_trial(
    {
        "learning_rate": 0.001,
        "batch_size": 128,
    }
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
)


## Save metrics and the best model

In [ ]:
results_df = (
    pd.DataFrame(trial_records)
    .sort_values("trial")
    .reset_index(drop=True)
)

results_df.to_csv(
    RESULTS_DIR / "all_100_trial_metrics.csv",
    index=False,
)

results_df["validation_mse"].to_csv(
    RESULTS_DIR / "Validation_loss.txt",
    index=False,
    header=False,
)

results_df["unseen_pearson"].to_csv(
    RESULTS_DIR / "Unseen_Pearson.txt",
    index=False,
    header=False,
)

results_df["unseen_spearman"].to_csv(
    RESULTS_DIR / "Unseen_Spearman.txt",
    index=False,
    header=False,
)

best_trial_number = study.best_trial.number

if best_global_trial != best_trial_number:
    raise RuntimeError(
        "Saved best weights do not match Optuna's best trial."
    )

best_params = study.best_trial.params

best_model = build_deepcas13_model(
    learning_rate=best_params[
        "learning_rate"
    ]
)

best_model.load_weights(
    best_weights_file
)

best_model.save(
    RESULTS_DIR / "best_DeepCas13.keras"
)

best_row = results_df.loc[
    results_df["trial"]
    == best_trial_number
].iloc[0]

summary = {
    "architecture_source": "deepcas13.py",
    "guides_containing_N_excluded_before_split": True,
    "architecture_modified": False,
    "preprocessing_modified": False,
    "original_training_differences": [
        "fixed train/validation/unseen split replaces 5-fold KFold ensemble",
        "100 trials replace one fixed training configuration",
        "validation set is used to select the reporting epoch",
    ],
    "authors_default_trial": {
        "trial": 0,
        "learning_rate": 0.001,
        "batch_size": 128,
        "epochs": 30,
    },
    "best_trial": int(best_trial_number),
    "best_params": best_params,
    "best_epoch": int(
        study.best_trial.user_attrs[
            "best_epoch"
        ]
    ),
    "validation_mse": float(
        best_row["validation_mse"]
    ),
    "unseen_pearson": float(
        best_row["unseen_pearson"]
    ),
    "unseen_spearman": float(
        best_row["unseen_spearman"]
    ),
}

with open(
    RESULTS_DIR / "best_trial_summary.json",
    "w",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )

display(results_df.head())
print("Best trial:", best_trial_number)
print(best_row)
print("Saved to:", RESULTS_DIR.resolve())
